# Notebook 6 (optional) — Real scanner data

*Part of **MRI Reconstruction from k-space to Deep Learning: A Hands-on Introduction***

Everything so far used simulated phantoms. In this notebook you apply the same methods to **real k-space data from an MRI scanner**: brain scans from the **M4Raw** dataset (Lyu et al., 2023), acquired on a 0.3 tesla scanner with a 4-channel head coil. M4Raw is openly licensed (CC BY 4.0) and can be downloaded one scan at a time, so there is no application to fill in.

Real data brings three new things:

- **Multiple receiver coils.** Each coil sees the head from a different side, and their images must be combined.
- **Noise.** Low-field scanners are affordable but noisy, so even the "fully sampled" reference image is noisy.
- **Real anatomy**, which looks nothing like ellipses, so the phantom-trained networks of Notebook 4 face a large distribution shift (Notebook 5).

**Learning objectives.** By the end of this notebook you will be able to:

1. Load raw multi-coil k-space from an HDF5 file and explain its dimensions.
2. Combine coil images into one image, with root-sum-of-squares and with estimated coil sensitivities.
3. Apply zero-filling, compressed sensing and an unrolled network to real undersampled data.
4. Train a reconstruction network on real data and compare it with one trained on phantoms.

**Prerequisites:** Notebooks 1–5.

**Time and resources:** about 90 minutes. The notebook downloads 25 scans (about 1 GB). Use a GPU runtime in Colab (*Runtime > Change runtime type > T4 GPU*); on a CPU, training takes about 15 minutes.

**Data credit:** M4Raw by Lyu, M., Mei, L., Huang, S., et al. (2023), *Scientific Data* 10, 264, licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/), downloaded from [huggingface.co/datasets/mylyu/M4Raw_brain](https://huggingface.co/datasets/mylyu/M4Raw_brain).

## 1. Setup and download

The first cell installs the Hugging Face download tool (usually already in Colab) and loads the libraries. The second repeats the helpers from earlier notebooks, including compressed sensing. The third defines the networks from Notebooks 4–5 and downloads their phantom-trained weights.

In [ ]:
%pip install -q huggingface_hub h5py   # usually already installed; ignore any message about restarting

import time
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download, list_repo_files
from skimage.metrics import structural_similarity

plt.rcParams["figure.dpi"] = 100
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # line colors, always used in this order
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
print(f"Using {device}.")

REPO, REVISION = "mylyu/M4Raw_brain", "v1.6.0"   # the M4Raw dataset on Hugging Face, fixed version

In [ ]:
# ---- From Notebooks 1-3 (now working on stacks of images too) ------------------
def fft2c(image):
    """Centered, orthonormal 2D FFT over the last two axes: image(s) -> k-space."""
    return np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(image, axes=(-2, -1)), norm="ortho"), axes=(-2, -1))


def ifft2c(kspace):
    """Centered, orthonormal inverse 2D FFT over the last two axes: k-space -> image(s)."""
    return np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(kspace, axes=(-2, -1)), norm="ortho"), axes=(-2, -1))


def show(images, titles, cmap="gray", vmin=None, vmax=None):
    """Show images side by side."""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(3.4 * n, 3.6))
    for ax, im, title in zip(np.atleast_1d(axes), images, titles):
        ax.set_title(title)
        ax.imshow(im, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def nrmse(recon, reference):
    """Normalized root-mean-square error: 0 is perfect, 1 is as large as the image itself."""
    return np.linalg.norm(recon - reference) / np.linalg.norm(reference)


def tidy(ax):
    """Quiet grid and axes for line plots."""
    ax.grid(alpha=0.25, linewidth=0.6)
    ax.spines[["top", "right"]].set_visible(False)


def variable_density_lines(N, R, center_fraction=0.08, decay=3, seed=0):
    """N // R lines: a fully sampled center, the rest random and more likely near the center."""
    rng = np.random.default_rng(seed)
    n_total = N // R
    n_center = int(round(N * center_fraction))
    lines = np.zeros(N, dtype=bool)
    lines[N // 2 - n_center // 2 : N // 2 - n_center // 2 + n_center] = True
    candidates = np.flatnonzero(~lines)
    distance = np.abs(candidates - N // 2) / (N / 2)
    probability = (1 - distance) ** decay + 1e-6
    lines[rng.choice(candidates, size=n_total - n_center, replace=False,
                     p=probability / probability.sum())] = True
    return lines


# ---- Compressed sensing from Notebook 3, for comparison --------------------------
SQRT2 = np.sqrt(2)


def haar2(x, levels=4):
    c = x.astype(complex).copy()
    n = c.shape[0]
    for _ in range(levels):
        a = c[:n, :n]
        a = np.hstack([(a[:, 0::2] + a[:, 1::2]) / SQRT2, (a[:, 0::2] - a[:, 1::2]) / SQRT2])
        a = np.vstack([(a[0::2, :] + a[1::2, :]) / SQRT2, (a[0::2, :] - a[1::2, :]) / SQRT2])
        c[:n, :n] = a
        n //= 2
    return c


def ihaar2(c, levels=4):
    x = c.copy()
    n = x.shape[0] // 2 ** (levels - 1)
    for _ in range(levels):
        a = x[:n, :n]
        h = n // 2
        b = np.empty_like(a)
        b[0::2, :] = (a[:h, :] + a[h:, :]) / SQRT2
        b[1::2, :] = (a[:h, :] - a[h:, :]) / SQRT2
        a2 = np.empty_like(a)
        a2[:, 0::2] = (b[:, :h] + b[:, h:]) / SQRT2
        a2[:, 1::2] = (b[:, :h] - b[:, h:]) / SQRT2
        x[:n, :n] = a2
        n *= 2
    return x


def soft_threshold(c, lam):
    magnitude = np.abs(c)
    return c * np.maximum(magnitude - lam, 0) / np.maximum(magnitude, 1e-12)


def cs_reconstruct(kspace_measured, lines, lam=0.01, n_iter=200, levels=4):
    n = kspace_measured.shape[0]
    coarse = np.zeros((n, n), dtype=bool)
    coarse[: n // 2**levels, : n // 2**levels] = True
    x = ifft2c(kspace_measured)
    for _ in range(n_iter):
        k = np.where(lines[:, None], kspace_measured, fft2c(x))
        x = ifft2c(k)
        w = haar2(x, levels)
        x = ihaar2(np.where(coarse, w, soft_threshold(w, lam)), levels)
    return x

In [ ]:
def conv_block(c_in, c_out):
    """Two 3x3 convolutions, each followed by ReLU."""
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1), nn.ReLU(inplace=True),
        nn.Conv2d(c_out, c_out, 3, padding=1), nn.ReLU(inplace=True),
    )


class UNet(nn.Module):
    """A small U-Net that predicts a correction to the zero-filled image."""

    def __init__(self, channels=2, base=16):
        super().__init__()
        # encoder: features at 128, 64 and 32 pixels
        self.enc1 = conv_block(channels, base)
        self.enc2 = conv_block(base, 2 * base)
        self.enc3 = conv_block(2 * base, 4 * base)
        # bottleneck at 16 pixels
        self.bottom = conv_block(4 * base, 8 * base)
        # decoder: upsample, join with the matching encoder features (skip connection), convolve
        self.up3, self.dec3 = nn.ConvTranspose2d(8 * base, 4 * base, 2, stride=2), conv_block(8 * base, 4 * base)
        self.up2, self.dec2 = nn.ConvTranspose2d(4 * base, 2 * base, 2, stride=2), conv_block(4 * base, 2 * base)
        self.up1, self.dec1 = nn.ConvTranspose2d(2 * base, base, 2, stride=2), conv_block(2 * base, base)
        self.out = nn.Conv2d(base, channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(F.max_pool2d(e1, 2))
        e3 = self.enc3(F.max_pool2d(e2, 2))
        b = self.bottom(F.max_pool2d(e3, 2))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return x + self.out(d1)          # residual: input + learned correction


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


def to_tensor(c):
    """Complex images (B, N, N) -> real tensor (B, 2, N, N)."""
    return torch.tensor(np.stack([c.real, c.imag], axis=1), dtype=torch.float32)


def to_complex(t):
    t = t.detach().cpu().numpy()
    return t[:, 0] + 1j * t[:, 1]


def tfft2c(x):
    return torch.fft.fftshift(torch.fft.fft2(torch.fft.ifftshift(x, dim=(-2, -1)), norm="ortho"), dim=(-2, -1))


def tifft2c(k):
    return torch.fft.fftshift(torch.fft.ifft2(torch.fft.ifftshift(k, dim=(-2, -1)), norm="ortho"), dim=(-2, -1))


def data_consistency(x, k_measured, mask):
    """As in Notebook 4, but the sampling mask is now an argument, so we can test other masks."""
    image = torch.complex(x[:, 0], x[:, 1])
    image = tifft2c(torch.where(mask, k_measured, tfft2c(image)))
    return torch.stack([image.real, image.imag], dim=1)


class SmallCNN(nn.Module):
    def __init__(self, channels=2, width=16, depth=5):
        super().__init__()
        layers = [nn.Conv2d(channels, width, 3, padding=1), nn.ReLU(inplace=True)]
        for _ in range(depth - 2):
            layers += [nn.Conv2d(width, width, 3, padding=1), nn.ReLU(inplace=True)]
        layers += [nn.Conv2d(width, channels, 3, padding=1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return x + self.net(x)


class UnrolledNet(nn.Module):
    def __init__(self, cascades=4):
        super().__init__()
        self.cnns = nn.ModuleList([SmallCNN() for _ in range(cascades)])

    def forward(self, x, k_measured, mask):
        for cnn in self.cnns:
            x = data_consistency(cnn(x), k_measured, mask)
        return x


def load_pretrained(model, filename):
    """Download the weights trained in Notebook 4 from the module's GitHub repository."""
    url = f"https://raw.githubusercontent.com/anupbagale/mri-recon-tutorial/main/weights/{filename}"
    urllib.request.urlretrieve(url, filename)
    model.load_state_dict(torch.load(filename, map_location=device))
    return model.to(device).eval()


# The two networks trained on phantoms in Notebook 4
phantom_unet = load_pretrained(UNet(), "unet_r4.pt")
phantom_unrolled = load_pretrained(UnrolledNet(), "unrolled_r4.pt")
print("Loaded the phantom-trained U-Net and unrolled network from Notebook 4.")

Now download the data: one T2-weighted scan from each of **20 training subjects** and **5 test subjects**. The test subjects come from M4Raw's separate validation split, so no person appears in both sets. Each file holds one complete scan of 18 slices.

In [ ]:
N_TRAIN_SUBJECTS, N_TEST_SUBJECTS = 20, 5

all_files = list_repo_files(REPO, repo_type="dataset", revision=REVISION)
# one T2-weighted scan (first repetition, "_T201") per subject; training and test subjects come from different splits
train_names = sorted(f for f in all_files if f.startswith("multicoil_train/") and f.endswith("_T201.h5"))[:N_TRAIN_SUBJECTS]
test_names = sorted(f for f in all_files if f.startswith("multicoil_val/") and f.endswith("_T201.h5"))[:N_TEST_SUBJECTS]


def download(name):
    """Download one scan (about 38 MB) from Hugging Face, or reuse it if it is already here."""
    return hf_hub_download(REPO, filename=name, repo_type="dataset", revision=REVISION)


start = time.time()
train_paths = [download(name) for name in train_names]
test_paths = [download(name) for name in test_names]
print(f"{len(train_paths)} training and {len(test_paths)} test scans ready ({time.time() - start:.0f} s)")
print("first test scan:", test_names[0])

## 2. What is in a raw data file?

M4Raw uses the same HDF5 layout as the large fastMRI dataset (Zbontar et al., 2018), so this code also works, with small changes, for fastMRI data.

In [ ]:
with h5py.File(test_paths[0], "r") as hf:
    print("keys:      ", list(hf.keys()))
    print("attributes:", dict(hf.attrs))
    volume_kspace = hf["kspace"][()]
    stored_rss = hf["reconstruction_rss"][()]

print("kspace:            ", volume_kspace.shape, volume_kspace.dtype,
      " (slices, coils, frequency encoding, phase encoding)")
print("reconstruction_rss:", stored_rss.shape)

- `kspace` holds the raw measurements: for every **slice**, every **coil**, a 2D grid of complex numbers. The third axis is the **frequency-encoding** direction (recorded in one readout) and the fourth the **phase-encoding** direction (one line per repetition, Notebook 2).
- `reconstruction_rss` is a reference image computed by the dataset authors.
- The attributes describe the scan, for example the contrast (`acquisition`).

## 3. Many coils, one image

Modern scanners receive the signal with several small coils placed around the head. Each coil measures its own k-space and sees mainly the part of the head closest to it.

> 🤔 **Predict:** What will the four coil images look like? And their k-spaces?

In [ ]:
def log_mag(kspace):
    """Log magnitude scaled to the brightest point (0) so faint and bright regions are both visible."""
    mag = np.abs(kspace)
    return np.log10(mag / mag.max() + 1e-6)


s = volume_kspace.shape[0] // 2              # a slice in the middle of the head
coil_kspace = volume_kspace[s]               # (coils, frequency encoding, phase encoding)
coil_images = ifft2c(coil_kspace)
n_coils = coil_kspace.shape[0]

show([log_mag(coil_kspace[c]) for c in range(n_coils)], [f"k-space, coil {c + 1}" for c in range(n_coils)])
show([np.abs(coil_images[c]) for c in range(n_coils)], [f"image, coil {c + 1}" for c in range(n_coils)])

Each coil image is bright near its coil and dark far from it: this is the coil's **sensitivity**. The simplest way to combine them is the **root-sum-of-squares (RSS)**, $\sqrt{\sum_c |x_c|^2}$, which is what the dataset authors stored in `reconstruction_rss`:

In [ ]:
rss = np.sqrt((np.abs(coil_images) ** 2).sum(axis=0))
print(f"Our RSS vs the file's reconstruction_rss: relative difference "
      f"{np.linalg.norm(rss - stored_rss[s]) / np.linalg.norm(stored_rss[s]):.1e}")
show([rss, stored_rss[s]], ["our root-sum-of-squares", "reconstruction_rss in the file"])

## 4. Which lines were actually measured?

> 🤔 **Predict:** Do all 256 phase-encode lines contain data?

In [ ]:
energy_pe = (np.abs(volume_kspace) ** 2).sum(axis=(0, 1, 2))   # energy in each phase-encode line
energy_fe = (np.abs(volume_kspace) ** 2).sum(axis=(0, 1, 3))   # energy in each frequency-encode row
acquired = energy_pe > 0
print(f"{acquired.sum()} of {acquired.size} phase-encode lines contain data")
print(f"{(energy_fe > 0).sum()} of {energy_fe.size} frequency-encode positions contain data")

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.semilogy(np.arange(acquired.size), energy_pe + 1e-30, color=COLORS[0], linewidth=2)
ax.set_xlabel("phase-encode line")
ax.set_ylabel("energy (log scale)")
ax.set_title("Energy per phase-encode line", loc="left")
ax.set_ylim(bottom=max(energy_pe[acquired].min() / 10, 1e-30))
tidy(ax)
plt.tight_layout()
plt.show()

**What to look for:**

- **Missing lines.** If fewer phase-encode lines than the image width contain data, the scanner measured only those and filled the rest with zeros to make a square image. This **zero-padding** makes the image look sharper on screen but adds no information. In this notebook, all sampling masks choose only among the **acquired** lines, so R = 4 means measuring a quarter of what the scanner really measured.
- **The energy profile.** As in Notebook 1, the energy is highest at the center of k-space and falls quickly. Towards the edges it flattens into a floor: there, the measurements are mostly **noise**. Real k-space edges carry much less image information than the noise-free phantoms suggested.

## 5. From four coils to one complex image

All our methods from Notebooks 2–5 work on a single complex image. RSS gives only a magnitude and throws the phase away. Instead, we estimate each coil's **sensitivity** from a low-resolution image made from the central k-space lines, and combine the coils with these weights. The result is one complex image with the scanner's real noise. Its phase is nearly flat inside the head: the head's smooth phase is absorbed into the estimated sensitivities. (This is similar to the "emulated single-coil" data that fastMRI provides; real multi-coil methods such as SENSE and GRAPPA use all coils during reconstruction, which is beyond this module.)

We also swap the two image axes, so that phase encoding runs **vertically**, as in all earlier notebooks. The heads therefore appear rotated, which does not matter for reconstruction.

In [ ]:
def load_volume(path):
    """Multi-coil k-space of one scan, with the two image axes swapped so that phase encoding
    runs vertically (along rows), as in Notebooks 2-5. The head therefore appears rotated."""
    with h5py.File(path, "r") as hf:
        return hf["kspace"][()].swapaxes(-1, -2)


def coil_combine(coil_kspace, n_center=24):
    """Combine coil images into one complex image, weighting each coil by its estimated sensitivity.
    The sensitivities are estimated from a low-resolution image made from the central k-space lines."""
    coil_images = ifft2c(coil_kspace)
    n = coil_kspace.shape[-2]
    center = slice(n // 2 - n_center // 2, n // 2 + n_center // 2)
    low = np.zeros_like(coil_kspace)
    low[:, center, :] = coil_kspace[:, center, :]
    low_images = ifft2c(low)
    sensitivities = low_images / (np.sqrt((np.abs(low_images) ** 2).sum(axis=0, keepdims=True)) + 1e-12)
    return (np.conj(sensitivities) * coil_images).sum(axis=0)


def prepare(paths, skip=3):
    """Coil-combined complex images of the middle slices of each scan, scaled so that the brightest
    0.5% of pixels are around 1. The first and last few slices contain little brain and are skipped."""
    images = []
    for path in paths:
        kspace = load_volume(path)
        for s in range(skip, kspace.shape[0] - skip):
            image = coil_combine(kspace[s])
            images.append(image / np.percentile(np.abs(image), 99.5))
    return np.stack(images).astype(np.complex64)


start = time.time()
train_images = prepare(train_paths)
test_images = prepare(test_paths)
print(f"{len(train_images)} training slices, {len(test_images)} test slices ({time.time() - start:.0f} s)")

example = test_images[len(test_images) // 2]
show([np.abs(example), np.angle(example)], ["coil-combined magnitude", "coil-combined phase"])

## 6. Undersampling real data

We undersample exactly as in Notebook 2: a variable-density mask with R = 4 and a fully sampled center, choosing only among the lines the scanner actually acquired.

> 🤔 **Predict:** Compared with the phantoms, will the zero-filled image look better or worse? Think about what real anatomy has that ellipses do not.

In [ ]:
def sampling_lines(acquired, R, center_fraction=0.08, decay=3, seed=0):
    """Variable-density sampling as in Notebook 2, but choosing only among the lines the scanner
    actually acquired. R is counted relative to the acquired lines."""
    rng = np.random.default_rng(seed)
    n = acquired.size
    available = np.flatnonzero(acquired)
    n_total = len(available) // R
    n_center = int(round(len(available) * center_fraction))
    lines = np.zeros(n, dtype=bool)
    lines[n // 2 - n_center // 2 : n // 2 - n_center // 2 + n_center] = True
    candidates = available[~lines[available]]
    distance = np.abs(candidates - n // 2) / (len(available) / 2)
    probability = np.clip(1 - distance, 0, None) ** decay + 1e-6
    lines[rng.choice(candidates, size=n_total - n_center, replace=False, p=probability / probability.sum())] = True
    return lines


R = 4
lines = sampling_lines(acquired, R)
print(f"Measuring {lines.sum()} of {acquired.sum()} acquired lines (R = {R})")

train_measured = fft2c(train_images) * lines[:, None]
test_measured = fft2c(test_images) * lines[:, None]
train_zf, test_zf = ifft2c(train_measured), ifft2c(test_measured)
train_truth, test_truth = np.abs(train_images), np.abs(test_images)

i = len(test_images) // 2
show([test_truth[i], np.abs(test_zf[i])], ["fully sampled (reference)", f"zero-filled, R = {R}"], vmin=0, vmax=1)

**What to look for:** the same kind of streaky, noise-like aliasing as in Notebook 2, now spread over real anatomy. Real brains have far more fine detail and texture than our ellipses, so more information sits in the outer k-space lines, and more of it is missing. Also compare the backgrounds: the reference image is not black outside the head, because real data contains noise everywhere.

## 7. Compressed sensing on real data

Compressed sensing needs no training, but λ must be chosen. Following Notebook 5's checklist, we tune λ on a few **training** slices and only then apply it to the test slices.

In [ ]:
def ssim(recon, reference):
    return structural_similarity(reference, recon, data_range=1.0)


def scores(recons, truths):
    """Mean NRMSE and SSIM over a set of slices."""
    return (np.mean([nrmse(r, t) for r, t in zip(recons, truths)]),
            np.mean([ssim(r, t) for r, t in zip(recons, truths)]))


# Choose λ on a few TRAINING slices, never on the test slices
tune = range(0, len(train_images), max(1, len(train_images) // 4))
lam_values = [0.001, 0.002, 0.005, 0.01, 0.02, 0.05]
lam_errors = []
for lam in lam_values:
    errors = [nrmse(np.abs(cs_reconstruct(train_measured[j], lines, lam)), train_truth[j]) for j in tune]
    lam_errors.append(np.mean(errors))
    print(f"λ = {lam:<6} NRMSE on {len(errors)} training slices: {lam_errors[-1]:.3f}")
best_lam = lam_values[int(np.argmin(lam_errors))]
print(f"Chosen λ = {best_lam}")

## 8. Training an unrolled network on real data

Notebook 5 showed that networks fail on images unlike their training data. So we train the same unrolled network as in Notebook 4, but now on the real training slices. One change: the loss compares **magnitudes**, because the reference is a complex image whose small leftover phase carries no anatomy.

In [ ]:
def to_tensors(zf, measured, truth):
    x = torch.tensor(np.stack([zf.real, zf.imag], axis=1), dtype=torch.float32)
    k = torch.tensor(measured, dtype=torch.complex64)
    y = torch.tensor(np.stack([truth, np.zeros_like(truth)], axis=1), dtype=torch.float32)
    return x, k, y


def train_unrolled(model, epochs, batch_size=8, lr=1e-3, seed=0, n_val=24):
    """Train an unrolled network on the real training slices; the last n_val slices
    (from the last two training subjects) are held out for validation."""
    torch.manual_seed(seed)
    model.to(device)
    mask = torch.tensor(lines[:, None], device=device)
    X, K, Y = to_tensors(train_zf, train_measured, train_truth)
    n_fit = len(X) - n_val
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train": [], "val": []}
    for epoch in range(epochs):
        start = time.time()
        model.train()
        order = np.random.default_rng(seed + epoch).permutation(n_fit)
        total = 0.0
        for b in range(0, n_fit, batch_size):
            idx = order[b : b + batch_size]
            out = model(X[idx].to(device), K[idx].to(device), mask)
            # compare magnitudes: the reference is complex, and its leftover phase carries no anatomy
            loss = F.l1_loss(torch.sqrt(out[:, 0] ** 2 + out[:, 1] ** 2 + 1e-12), Y[idx, 0].to(device))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(idx)
        model.eval()
        with torch.no_grad():
            out = model(X[n_fit:].to(device), K[n_fit:].to(device), mask)
            val_loss = F.l1_loss(torch.sqrt(out[:, 0] ** 2 + out[:, 1] ** 2 + 1e-12), Y[n_fit:, 0].to(device)).item()
        history["train"].append(total / n_fit)
        history["val"].append(val_loss)
        print(f"epoch {epoch + 1}/{epochs}: train loss {total / n_fit:.4f}, validation loss {val_loss:.4f}  ({time.time() - start:.0f} s)")
    return history


def predict(model, zf, measured, mask_lines=None):
    """Magnitude reconstructions from a trained network (U-Net or unrolled)."""
    mask_lines = lines if mask_lines is None else mask_lines
    model.eval()
    x = torch.tensor(np.stack([zf.real, zf.imag], axis=1), dtype=torch.float32).to(device)
    k = torch.tensor(measured, dtype=torch.complex64).to(device)
    mask = torch.tensor(mask_lines[:, None], device=device)
    outputs = []
    with torch.no_grad():
        for b in range(0, len(x), 8):
            out = model(x[b:b + 8], k[b:b + 8], mask) if isinstance(model, UnrolledNet) else model(x[b:b + 8])
            outputs.append(torch.sqrt(out[:, 0] ** 2 + out[:, 1] ** 2).cpu().numpy())
    return np.concatenate(outputs)

In [ ]:
USE_SAVED = False   # set to True to reuse weights saved by an earlier run of this notebook

real_unrolled = UnrolledNet().to(device)
if USE_SAVED:
    real_unrolled.load_state_dict(torch.load("unrolled_m4raw_r4.pt", map_location=device))
else:
    history = train_unrolled(real_unrolled, epochs=10)
    torch.save(real_unrolled.state_dict(), "unrolled_m4raw_r4.pt")

    fig, ax = plt.subplots(figsize=(5.5, 3.5))
    epochs = np.arange(1, len(history["train"]) + 1)
    ax.plot(epochs, history["train"], "o-", color=COLORS[0], linewidth=2, markersize=8, label="training slices")
    ax.plot(epochs, history["val"], "o-", color=COLORS[1], linewidth=2, markersize=8, label="validation slices")
    ax.set_xlabel("epoch")
    ax.set_ylabel("L1 loss")
    ax.set_title("Training on real data", loc="left")
    ax.set_xticks(epochs)
    ax.set_ylim(bottom=0)
    ax.legend(frameon=False)
    tidy(ax)
    plt.tight_layout()
    plt.show()

## 9. The comparison

We compare five reconstructions on the same test slices (every third one, to keep compressed sensing quick):

- zero-filled and compressed sensing, which need no training;
- the U-Net and unrolled network **trained on phantoms** in Notebook 4;
- the unrolled network **trained on real data** in Section 8.

> 🤔 **Predict:** Rank the five methods. Where will the phantom-trained networks end up?

In [ ]:
subset = range(0, len(test_images), 3)
truths = test_truth[list(subset)]
results = {
    "zero-filled": np.abs(test_zf[list(subset)]),
    "compressed sensing": np.stack([np.abs(cs_reconstruct(test_measured[j], lines, best_lam)) for j in subset]),
    "U-Net (phantoms)": predict(phantom_unet, test_zf[list(subset)], test_measured[list(subset)]),
    "unrolled (phantoms)": predict(phantom_unrolled, test_zf[list(subset)], test_measured[list(subset)]),
    "unrolled (real data)": predict(real_unrolled, test_zf[list(subset)], test_measured[list(subset)]),
}

print(f"{'method':<24}{'NRMSE ↓':>9}{'SSIM ↑':>9}   ({len(truths)} test slices from {N_TEST_SUBJECTS} subjects)")
for name, recon in results.items():
    e, s_ = scores(recon, truths)
    print(f"{name:<24}{e:>9.3f}{s_:>9.3f}")

In [ ]:
j = len(truths) // 2
names = list(results)
show([truths[j]] + [results[n][j] for n in names[:2]], ["reference"] + names[:2], vmin=0, vmax=1)
show([results[n][j] for n in names[2:]], names[2:], vmin=0, vmax=1)
show([results[n][j] - truths[j] for n in names], [f"error: {n}" for n in names], cmap="RdBu_r", vmin=-0.3, vmax=0.3)

**Questions to answer from your results:**

1. **Phantoms vs real training.** How do the two phantom-trained networks compare with zero-filling? Notebook 5 showed that networks can do worse than zero-filling on unfamiliar images. Does that happen here, and does data consistency help the phantom-trained unrolled network?
2. **The real-data network vs compressed sensing.** With only 20 training subjects and a few minutes of training, does the unrolled network beat compressed sensing on both NRMSE and SSIM?
3. **The error maps.** Are the remaining errors mostly noise-like, or do they follow anatomical edges? Which method's error map shows the least structure?

Write down your answers, then read Section 10: the reference image itself is noisy, which affects every number in the table.

## 10. What the reference cannot tell you

Look closely at the **reference** image: it is noisy, because a 0.3 T scanner receives a weak signal. Our error measures compare every method with this noisy reference, so a method that **removes noise** is penalized for it, even though a radiologist might prefer its image. The "ground truth" in real data is never perfectly true.

M4Raw has a unique way around this: each subject was scanned **several times** with the same settings. Averaging the repetitions gives a reference with much less noise. Exercise 3 uses this to re-evaluate the methods.

## Exercises

Solutions are in `solutions/06_solutions.ipynb`.

### Exercise 1: higher acceleration on real data

Repeat the comparison of zero-filling, compressed sensing and the real-data unrolled network at R = 6 and R = 8, on 10 test slices, *without retraining* the network (it was trained at R = 4). How fast does each method degrade?

<details><summary>Hint</summary>

Build new lines with <code>sampling_lines(acquired, 6)</code>, recompute the measured k-space and zero-filled images for the test slices, and pass the new lines to both <code>cs_reconstruct</code> and <code>predict(..., mask_lines=new_lines)</code>.
</details>

In [ ]:
# Exercise 1: your code here


### Exercise 2: transfer learning

Instead of training from scratch, start from the **phantom-trained** unrolled network and train it on the real data for 3 epochs (`train_unrolled(model, epochs=3)`). Compare it with a network trained from scratch for the same 3 epochs. Does starting from phantoms help?

<details><summary>Hint</summary>

Copy the phantom network first (<code>copy.deepcopy(phantom_unrolled)</code>) so the original stays unchanged.
</details>

In [ ]:
# Exercise 2: your code here


### Exercise 3: a cleaner reference

For the first test subject, download the other repetitions of the same T2 scan (replace `_T201` with `_T202` and `_T203` in the file name), coil-combine the same slices, and average the three repetitions. Scale every repetition by the **same** factor as the first one. Then:

1. Show a single-repetition image next to the average. How much does the noise drop?
2. Recompute NRMSE and SSIM of each method's reconstruction of that subject against the averaged reference. Which methods gain the most, and why?

<details><summary>Hint</summary>

Check the file names exist in <code>all_files</code> before downloading. Reuse <code>load_volume</code> and <code>coil_combine</code>, and divide by the same percentile value you used for repetition 1.
</details>

In [ ]:
# Exercise 3: your code here


### Exercise 4: a different contrast

M4Raw also contains **FLAIR** scans (`_FLAIR01.h5`) of the same subjects. Prepare the FLAIR scans of the test subjects, undersample them with the same mask, and compare compressed sensing with the unrolled network trained on **T2** images. Does the network still beat compressed sensing on a contrast it never saw?

In [ ]:
# Exercise 4: your code here


## Summary

- Raw MRI data is **multi-coil k-space**: one complex grid per coil and slice. Coils are combined with root-sum-of-squares, or with estimated **sensitivities** to keep a complex image.
- Scanners often measure fewer phase-encode lines than the image size and fill the rest with zeros; acceleration factors must count only the **acquired** lines.
- Compressed sensing works on real data without training, but λ must be tuned on separate data.
- Networks trained on phantoms face a large **distribution shift** on real anatomy; training on real data from the same kind of scanner is essential.
- In real data the reference image is itself **noisy**, so error measures must be read with care.

**Where to go next:** multi-coil reconstruction methods (SENSE, GRAPPA and multi-coil unrolled networks such as the variational network), the larger fastMRI dataset, and open-source toolboxes such as SigPy and BART.

## References

1. Lyu, M., Mei, L., Huang, S., et al. (2023). M4Raw: A multi-contrast, multi-repetition, multi-channel MRI k-space dataset for low-field MRI research. *Scientific Data*, 10, 264.
2. Zbontar, J., et al. (2018). fastMRI: An open dataset and benchmarks for accelerated MRI. *arXiv:1811.08839*.
3. Roemer, P. B., Edelstein, W. A., Hayes, C. E., Souza, S. P., & Mueller, O. M. (1990). The NMR phased array. *Magnetic Resonance in Medicine*, 16(2), 192–225.
4. Pruessmann, K. P., Weiger, M., Scheidegger, M. B., & Boesiger, P. (1999). SENSE: Sensitivity encoding for fast MRI. *Magnetic Resonance in Medicine*, 42(5), 952–962.
5. Sriram, A., et al. (2020). End-to-end variational networks for accelerated MRI reconstruction. *MICCAI 2020*, LNCS 12262, 64–73.

---
*Text and figures: CC-BY 4.0. Code: MIT License. M4Raw data: CC BY 4.0, Lyu et al. (2023).*